# Southeast Asian Sex Trafficking Network: Analytic Notebook

**Deliverable 7 supporting analysis: GEOG 594A**

This notebook documents the end-to-end data pipeline and network analysis used to identify structural chokepoints in the Southeast Asian sex-trafficking network. It is written to be reproducible and self-contained: each step states what it does, why it is necessary, and what result confirms it worked.

**Data.** Counter Trafficking Data Collaborative (CTDC) Global Synthetic Dataset, 2002-2024. The data are differentially private synthetic records: aggregate patterns are preserved, but individual records are not ground truth.

**Method.** The dataset is filtered to sexual-exploitation cases connected to seven Southeast Asian ORIGIN countries, aggregated into an origin-destination edge list, and represented as a directed weighted graph. Betweenness centrality is then computed to identify countries occupying structural transit positions in the network.

**Scope.** This analysis includes records where a Southeast Asian country is either the origin or the destination of exploitation, capturing each country's dual role as both a source and a transit or destination point.

## 1. Environment and libraries

The analysis uses two primary open-source libraries: pandas for tabular data handling and NetworkX for graph construction and centrality computation. Library versions are recorded below to support reproducibility.

In [1]:
import pandas as pd
import networkx as nx

print("pandas  :", pd.__version__)
print("networkx:", nx.__version__)

pandas : 2.3.3
networkx: 3.5


## 2. Loading the dataset

The full CTDC dataset is loaded into a pandas DataFrame. Its dimensions are reported to confirm the complete record set is present before any filtering is applied. The `low_memory=False` option ensures each column is assigned a single consistent data type across the entire file.

In [2]:
df = pd.read_csv("data/ctdc_global_synthetic_data_v2026.csv", low_memory=False)

print("Records:", df.shape[0])
print("Columns:", df.shape[1])

Records: 268515
Columns: 27


## 3. Restricting to sexual-exploitation cases

The dataset records multiple forms of exploitation. This analysis concerns sexual exploitation, identified by the `isSexualExploit` field, which marks qualifying cases with a value of 1 and is otherwise empty. Records not flagged for sexual exploitation are excluded.

In [3]:
sex_df = df[df["isSexualExploit"] == 1]

print("Sexual-exploitation records:", sex_df.shape[0])

Sexual-exploitation records: 100390


## 4. Population of interest and analytic scope

The analysis focuses on seven Southeast Asian ORIGIN countries: the Philippines, Thailand, Indonesia, Vietnam, Cambodia, Myanmar, and Laos (ISO3 codes PHL, THA, IDN, VNM, KHM, MMR, LAO).

Two scoping options are possible:

1. **Origin only**: records where a Southeast Asian country is the victim's citizenship.
2. **Origin or destination**: records where a Southeast Asian country is either the origin (citizenship) or the destination (country of exploitation).

This analysis uses the **origin-or-destination** scope. The rationale is methodological. A network built on origin-only records represents the seven countries almost exclusively as points of departure: they gain outbound connections but virtually no inbound ones. Betweenness centrality measures how often a node lies on the shortest paths between other nodes, which requires a node to have both inbound and outbound connections. Restricting to origins therefore structurally suppresses the transit signal this analysis is designed to detect. Including records where a Southeast Asian country is the destination restores inbound connections and allows each country's dual role: as both a source and a transit or destination point: to be represented.

The counts below characterize the population of interest under each scope. This analysis uses the origin-or-destination total.

In [4]:
sea_codes = ["PHL", "THA", "IDN", "VNM", "KHM", "MMR", "LAO"]

origin = sex_df[sex_df["citizenship"].isin(sea_codes)]
dest   = sex_df[sex_df["CountryOfExploitation"].isin(sea_codes)]
either = sex_df[sex_df["citizenship"].isin(sea_codes) | sex_df["CountryOfExploitation"].isin(sea_codes)]

print("SEA as origin:               ", origin.shape[0])
print("SEA as destination:          ", dest.shape[0])
print("SEA as origin or destination:", either.shape[0])

SEA as origin:                3664
SEA as destination:           1822
SEA as origin or destination: 4315


## 5. Constructing the corridor edge list

A network edge requires two endpoints: an origin and a destination. Records missing either the country of citizenship or the country of exploitation cannot be placed on a corridor and are excluded from network construction. The retained records are then aggregated into origin-destination pairs, each weighted by the number of victims recorded on that corridor.

In [5]:
complete = either.dropna(subset=["citizenship", "CountryOfExploitation"])

print("Records with both endpoints present:", complete.shape[0])

Records with both endpoints present: 2267


### Aggregation and removal of domestic corridors

Records are grouped by origin-destination country pair and counted, producing a weighted edge list in which each row is one directional corridor and its victim volume. Self-loops: corridors whose origin and destination are the same country: represent domestic (intra-country) exploitation rather than cross-border movement. Because this analysis concerns cross-border network structure and chokepoints, self-loops are removed from the graph; their volume is retained separately for reporting.

In [6]:
edges = complete.groupby(["citizenship", "CountryOfExploitation"]).size().reset_index(name="weight")

self_loops = edges[edges["citizenship"] == edges["CountryOfExploitation"]]
edges = edges[edges["citizenship"] != edges["CountryOfExploitation"]]

print("Total corridors (incl. self-loops):", len(self_loops) + len(edges))
print("Domestic self-loops removed:", len(self_loops), "corridors,", self_loops["weight"].sum(), "records")
print("Cross-border corridors:", len(edges), "|", int(edges["weight"].sum()), "records")

Total corridors (incl. self-loops): 162
Domestic self-loops removed: 4 corridors, 996 records
Cross-border corridors: 158 | 1271 records


## 6. Constructing the directed weighted network

The cross-border edge list is represented as a directed, weighted graph. Nodes are countries; directed edges are corridors from origin to destination; edge weights are victim volumes. Direction is retained because trafficking flows are directional: an origin-to-destination corridor is not equivalent to its reverse.

In [7]:
G = nx.from_pandas_edgelist(
    edges,
    source="citizenship",
    target="CountryOfExploitation",
    edge_attr="weight",
    create_using=nx.DiGraph()
)

print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())

Nodes: 71
Edges: 158


## 7. Betweenness centrality

Betweenness centrality measures how frequently a node lies on the shortest paths between all pairs of other nodes. A country with high betweenness occupies a structural transit position: many corridors between other countries pass through it. This is the primary measure used to identify chokepoints.

Betweenness is computed in two forms:

- **Unweighted**, treating every corridor as a single step regardless of victim volume. This reflects pure network topology.
- **Weighted**, accounting for corridor volume. NetworkX interprets edge weight as distance (cost to traverse), so victim volume is inverted (distance = 1 / volume) before computation. Under this transformation, higher-volume corridors are treated as shorter, preferred paths, consistent with the assumption that flow concentrates on higher-volume routes.

Because betweenness is a normalized structural measure, it does not by itself indicate how much data underlies each score. Each unweighted score below is therefore reported alongside the node's record volume (total victims on all corridors touching that country), so that structurally robust results can be distinguished from low-volume artifacts.

In [8]:
bc_unwtd = nx.betweenness_centrality(G)

top_unwtd = sorted(bc_unwtd.items(), key=lambda x: x[1], reverse=True)[:8]
print("Betweenness (unweighted), top 8, with record volume:")
for country, score in top_unwtd:
    records = int(G.in_degree(country, weight="weight") + G.out_degree(country, weight="weight"))
    print(f"  {country}: {score:.4f}   ({records} records)")

Betweenness (unweighted), top 8, with record volume:
  KHM: 0.1186   (154 records)
  THA: 0.1095   (223 records)
  PHL: 0.1086   (298 records)
  IDN: 0.1041   (127 records)
  VNM: 0.0916   (518 records)
  MMR: 0.0467   (118 records)
  KEN: 0.0312   (10 records)
  ROU: 0.0200   (7 records)


### 8. Weighted betweenness

To account for corridor volume, victim counts are converted to distances (distance = 1 / volume) so that higher-volume corridors are treated as shorter, preferred paths. Betweenness is then recomputed using these distances. Comparing the weighted and unweighted rankings tests whether the identified transit positions are robust to how corridor volume is treated.

In [9]:
for u, v, d in G.edges(data=True):
    d["distance"] = 1 / d["weight"]

bc_wtd = nx.betweenness_centrality(G, weight="distance")

top_wtd = sorted(bc_wtd.items(), key=lambda x: x[1], reverse=True)[:8]
print("Betweenness (weighted by volume), top 8, with record volume:")
for country, score in top_wtd:
    records = int(G.in_degree(country, weight="weight") + G.out_degree(country, weight="weight"))
    print(f"  {country}: {score:.4f}   ({records} records)")

Betweenness (weighted by volume), top 8, with record volume:
  KHM: 0.1716   (154 records)
  THA: 0.1653   (223 records)
  IDN: 0.1290   (127 records)
  VNM: 0.1179   (518 records)
  PHL: 0.0963   (298 records)
  COL: 0.0716   (11 records)
  MMR: 0.0683   (118 records)
  BLR: 0.0354   (8 records)


## 9. Results for the countries of interest

Interpretation is constrained to the seven Southeast Asian ORIGIN countries, the population of interest, each supported by sufficient record volume (see Section 8). The table below summarizes their centrality and volume, sorted by weighted betweenness. Peripheral non-SEA nodes are excluded from interpretation as low-volume dyadic-chaining artifacts.

In [10]:
sea_summary = []
for c in sea_codes:
    records = int(G.in_degree(c, weight="weight") + G.out_degree(c, weight="weight"))
    sea_summary.append({
        "country": c,
        "betweenness_unwtd": round(bc_unwtd[c], 4),
        "betweenness_wtd": round(bc_wtd[c], 4),
        "records": records,
        "in_corridors": G.in_degree(c),
        "out_corridors": G.out_degree(c),
    })

sea_summary = pd.DataFrame(sea_summary).sort_values("betweenness_wtd", ascending=False)
print(sea_summary.to_string(index=False))


country  betweenness_unwtd  betweenness_wtd  records  in_corridors  out_corridors
    KHM             0.1186           0.1716      154            14             18
    THA             0.1095           0.1653      223            15             16
    IDN             0.1041           0.1290      127            11             19
    VNM             0.0916           0.1179      518            10             17
    PHL             0.1086           0.0963      298            13             18
    MMR             0.0467           0.0683      118             6              7
    LAO             0.0000           0.0000        8             0              5


## 10. Exporting results

The centrality results for the countries of interest are written to CSV for use in downstream mapping (ArcGIS join) and reporting.

In [11]:
import os

os.makedirs("outputs", exist_ok=True)
out_path = "outputs/GEOG594A_D7_CentralityResults.csv"
sea_summary.to_csv(out_path, index=False)

print("Saved:", out_path)

Saved: outputs/GEOG594A_D7_CentralityResults.csv


## 11. Exporting spatial join tables

Three tables are exported to support mapping in ArcGIS Pro, each keyed by ISO3 country code for joining to a country boundary layer:

- **Edge list** (origin, destination, weight): corridor-level data for flow maps.
- **Outbound records by origin**: total victims departing each country, for an origin choropleth.
- **Inbound records by destination**: total victims exploited in each country, for a destination choropleth.

Combined with the centrality results (Section 10), these support the full set of maps for the analysis.

In [12]:
# Corridor edge list (for flow maps)
edges_out = edges.rename(columns={"citizenship": "origin", "CountryOfExploitation": "destination"})
edges_out.to_csv("outputs/GEOG594A_D7_EdgeList.csv", index=False)

# Outbound volume by origin (origin choropleth)
origin_vol = edges.groupby("citizenship")["weight"].sum().reset_index()
origin_vol.columns = ["ISO3", "outbound_records"]
origin_vol.to_csv("outputs/GEOG594A_D7_OutboundByOrigin.csv", index=False)

# Inbound volume by destination (destination choropleth)
dest_vol = edges.groupby("CountryOfExploitation")["weight"].sum().reset_index()
dest_vol.columns = ["ISO3", "inbound_records"]
dest_vol.to_csv("outputs/GEOG594A_D7_InboundByDestination.csv", index=False)

print("Saved 3 files:")
print("  edge list           :", len(edges_out), "corridors")
print("  outbound by origin  :", len(origin_vol), "countries")
print("  inbound by destination:", len(dest_vol), "countries")

Saved 3 files:
  edge list          : 158 corridors
  outbound by origin : 41 countries
  inbound by destination: 51 countries


## 12. Network redundancy and interdiction feasibility

The centrality results identify which countries occupy structural transit positions. They do not establish whether disrupting those positions would meaningfully interrupt flow. A network in which every transit route has a competing direct route is robust to interdiction: removing a hub diverts flow rather than stopping it. A network without such alternatives is vulnerable: removing a hub breaks the route. This distinction determines whether the analysis supports an actionable recommendation.

Two descriptive measures address this:

- **Shortest-path length distribution.** A path of length 1 is a direct origin-to-destination corridor that passes through no intermediate country. A path of length 2 or greater passes through at least one intermediate node. The proportion of paths exceeding length 1 indicates how much of the network's connectivity depends on transit through other countries.

- **Global transitivity.** Transitivity measures how often, where a two-step path exists between two nodes, a direct edge also connects them. High transitivity indicates redundant pathways and therefore robustness to disruption. Low transitivity indicates that intermediate nodes are not bypassable.

These measures were computed on the Deliverable 6 network, which used an origin-only scope (52 nodes, 1,102 cross-border records), yielding a transitivity of 0.037 and a path-length distribution of 34.6 percent at length 1, 35.3 percent at length 2, 21.8 percent at length 3, and 8.3 percent at length 4. The scope broadening applied in this analysis adds inbound corridors to the seven Southeast Asian origin countries, which is the specific structural change expected to alter path composition. The measures are therefore recomputed here using identical code, so that the two networks can be compared directly and the effect of the scope change assessed empirically rather than assumed.

In [13]:
from collections import Counter

# --- identical method to the D6 computation, applied to the D7 graph ---
transitivity = nx.transitivity(G)

path_lengths = []
for source in G.nodes():
    lengths = nx.single_source_shortest_path_length(G, source)
    path_lengths.extend([d for d in lengths.values() if d > 0])

dist = Counter(path_lengths)
total = sum(dist.values())

print("D7 network:", G.number_of_nodes(), "nodes,", G.number_of_edges(), "edges")
print("Global transitivity:", round(transitivity, 3), "   (D6 was 0.037)")

print("\nShortest-path length distribution:")
d6_reference = {1: 34.6, 2: 35.3, 3: 21.8, 4: 8.3}
for length in sorted(dist):
    pct = dist[length] / total * 100
    ref = d6_reference.get(length)
    ref_txt = f"   (D6: {ref}%)" if ref is not None else ""
    print(f"  length {length}: {dist[length]} paths ({pct:.1f}%){ref_txt}")

multi_hop = sum(dist[k] for k in dist if k > 1) / total * 100
print(f"\nPaths requiring at least one intermediate country: {multi_hop:.1f}%   (D6: 65.4%)")

D7 network: 71 nodes, 158 edges
Global transitivity: 0.056    (D6 was 0.037)

Shortest-path length distribution:
  length 1: 158 paths (7.6%)   (D6: 34.6%)
  length 2: 898 paths (43.4%)   (D6: 35.3%)
  length 3: 581 paths (28.1%)   (D6: 21.8%)
  length 4: 433 paths (20.9%)   (D6: 8.3%)

Paths requiring at least one intermediate country: 92.4%   (D6: 65.4%)


## 13. Domestic and cross-border composition by origin country

Section 5 removed domestic corridors from the network so that centrality would reflect cross-border structure only. Those records were retained for separate reporting rather than discarded. This section quantifies them.

For each of the five hypothesis countries, records are classified by what the origin country's cases represent:

- **No destination recorded.** The country of exploitation is missing, so the record cannot be placed on a corridor in either direction.
- **Domestic.** The country of exploitation is the same as the country of citizenship. The victim was exploited without crossing a border.
- **Cross-border outbound.** The country of exploitation differs from the country of citizenship, producing a directed corridor in the network.

The resulting composition serves two purposes. First, it reconciles the two record counts used in this project: demographic analysis is conducted over all records of a given origin, while network analysis is restricted to records forming cross-border corridors. Second, the proportion of a country's exploitation that crosses a border is itself diagnostic evidence for the chokepoint question, since a transit role requires border crossing by definition.

In [14]:
# Composition of each SEA country's origin records: domestic vs cross-border
composition = []
for c in sea_codes:
    sub = sex_df[sex_df["citizenship"] == c]
    n = len(sub)
    no_dest     = sub["CountryOfExploitation"].isna().sum()
    domestic    = (sub["CountryOfExploitation"] == c).sum()
    crossborder = n - no_dest - domestic
    classifiable = n - no_dest

    composition.append({
        "ISO3": c,
        "total_origin_records": n,
        "no_destination": int(no_dest),
        "destination_miss%": round(100 * no_dest / n, 1) if n else 0.0,
        "domestic": int(domestic),
        "cross_border_out": int(crossborder),
        "pct_domestic_of_known": round(100 * domestic / classifiable, 1) if classifiable else 0.0,
    })

composition = pd.DataFrame(composition).sort_values("pct_domestic_of_known", ascending=False)
print(composition.to_string(index=False))

composition.to_csv("outputs/GEOG594A_D7_DomesticComposition.csv", index=False)
print("\nSaved: outputs/GEOG594A_D7_DomesticComposition.csv")

ISO3  total_origin_records  no_destination  destination_miss%  domestic  cross_border_out  pct_domestic_of_known
 KHM                   960             212               22.1       678                70                   90.6
 PHL                   977             520               53.2       182               275                   39.8
 IDN                   317             169               53.3        42               106                   28.4
 VNM                  1070             493               46.1        94               483                   16.3
 THA                   300             152               50.7         0               148                    0.0
 MMR                    26              14               53.8         0                12                    0.0
 LAO                    14               6               42.9         0                 8                    0.0

Saved: outputs/GEOG594A_D7_DomesticComposition.csv
